# Aligning Async Streams

Real-world data sources tick at their own rate: one stream might update every
second, another every five minutes. Aligning them by row number is wrong -
it implicitly assumes they tick in sync. screamer's multi-stream layer uses an
explicit **index** (an integer timestamp or sequence number) so that
`combine_latest` can forward-fill each input's last seen value and emit one
aligned row whenever any input ticks. The carry is fully causal: it propagates
the last *observed* value, never a future one.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from screamer import combine_latest, merge

# Two async feeds with different timestamps (keys)
a_k = np.array([1, 3, 5, 7, 9], dtype=np.int64)
a_v = np.array([10., 11., 12., 11., 13.])

b_k = np.array([2, 4, 6, 8], dtype=np.int64)
b_v = np.array([5., 6., 5.5, 6.5])

## The stream index model

A **stream** in screamer is values plus an optional **index**: a monotonically
non-decreasing `int64` array that locates each value in a shared timeline.
The index is an ordering coordinate (nanoseconds, sequence numbers, bar
positions - any metric). Operations respect the index rather than row positions,
so two streams can be combined correctly even when their tick rates differ.

When no index is given, the streams are assumed to be **positional** (aligned
clocks) and must have equal length. Provide an index whenever the streams tick
on different clocks.


In [ ]:
print("Stream a - index:", a_k, "  values:", a_v)
print("Stream b - index:", b_k, "  values:", b_v)
print()
print("Stream a ticks at odd indices; stream b at even indices - no shared points.")
print("Row-number alignment would pair a[0]=10 with b[0]=5, but they are at")
print("indices 1 and 2 - a full tick apart.")


## `combine_latest` as-of join

`combine_latest` merges N streams by forward-filling each input's last known
value and emitting one aligned row on every distinct index. The default
`emit="when_all"` waits until every input has been seen at least once before
producing output, so the first row always has a real observation from every
stream - no cold-start NaNs. Once aligned, the two columns can be fed
directly into any functor - unpack values-first in eager array code:

```python
aligned, idx = combine_latest(a_v, b_v, index=[a_k, b_k])
result = RollingCorr(10)(aligned)     # aligned is the (T, 2) values array
```

(Inside a `Dag`, `combine_latest` returns a Node, so the shorter
`RollingCorr(10)(combine_latest(a, b))` form works there.)


In [ ]:
# combine_latest: emit whenever EITHER ticks, carrying each input's last value
aligned, idx = combine_latest(a_v, b_v, index=[a_k, b_k])  # emit="when_all" default
spread = aligned[:, 0] - aligned[:, 1]

print("Aligned index:", idx)
print("aligned[:, 0] (stream a carried):", aligned[:, 0])
print("aligned[:, 1] (stream b carried):", aligned[:, 1])
print("spread (a - b)                  :", spread)

# Spread is causal: each spread value uses only events observed so far
assert len(idx) == 8   # 5 ticks from a + 4 ticks from b - 1 skipped (before warm-up)
assert not np.isnan(spread).any(), "when_all guarantees no NaN in spread"


## `emit="when_all"` vs `"on_any"`

`emit="on_any"` starts emitting from the very first event. Streams not yet
seen carry `NaN` until their first observation arrives. This is useful when
you need the earliest possible output and can handle the initial NaN period
(e.g. via `dropna` downstream).


In [ ]:
# on_any emits from the first event (NaN for not-yet-seen inputs)
aligned_any, idx_any = combine_latest(a_v, b_v, index=[a_k, b_k], emit="on_any")

print("on_any index :", idx_any)
print("on_any aligned:")
print(aligned_any)

# First row: only stream a has ticked; stream b is NaN
assert np.isnan(aligned_any[0, 1]), "stream b NaN before its first tick"
# After index=2, both streams have ticked - no more NaN
assert not np.isnan(aligned_any[2:]).any()

# when_all produces a strict subset of on_any (skips the cold-start rows)
np.testing.assert_array_equal(idx, idx_any[1:])       # when_all drops index=1
np.testing.assert_array_equal(aligned, aligned_any[1:])  # values are the same after warm-up


## `merge` - one sorted tagged stream

`merge` interleaves N streams into a single index-sorted `(values, sources,
index)` triple. The `sources` array records which input each event came
from. This is the building block for `pace` (notebook 08) and `split`
(notebook 09); it is also useful for feeding heterogeneous events into a
single downstream processor.


In [ ]:
# merge: interleave into one index-sorted, source-tagged stream
mv, ms, mk = merge(a_v, b_v, index=[a_k, b_k])

print("merged index  :", mk)
print("merged values :", mv)
print("sources (0=a, 1=b):", ms)

assert (np.diff(mk) >= 0).all(), "merge output is globally index-sorted"
assert len(mk) == len(a_k) + len(b_k), "merge concatenates all events"


## Plotting the aligned spread

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8, 5), sharex=True)

# Raw streams
axes[0].step(a_k, a_v, where="post", label="stream a", color="steelblue")
axes[0].step(b_k, b_v, where="post", label="stream b", color="coral")
axes[0].set_ylabel("price")
axes[0].legend()
axes[0].set_title("Raw async streams")

# Aligned spread
axes[1].step(idx, spread, where="post", label="spread (a-b, when_all)", color="purple")
axes[1].axhline(0, color="gray", linewidth=0.8, linestyle="--")
axes[1].set_xlabel("index (logical time)")
axes[1].set_ylabel("spread")
axes[1].legend()
axes[1].set_title("Spread computed on aligned values")

plt.tight_layout()
plt.show()


**Takeaways**

- A stream is values plus an optional **index** (ordering coordinate, not a
  row label). Alignment respects the index, so different tick rates are handled
  automatically.
- `combine_latest` is a causal as-of join: it forward-fills each input's last
  *observed* value - no lookahead, no future information. One row per distinct
  index (same-index events coalesce).
- `emit="when_all"` (default) suppresses output until every stream has at least
  one observation; `emit="on_any"` emits from the first tick with NaN for
  cold inputs.
- `merge` interleaves N streams into one sorted tagged stream; `split` inverts
  it (see notebook 09).
- Return is values-first: `aligned, idx = combine_latest(a_v, b_v, index=[a_k, b_k]);
  RollingCorr(10)(aligned)` gives a rolling correlation between two async
  streams with no extra wiring. (Inside a `Dag` the shorter
  `RollingCorr(10)(combine_latest(a, b))` form works because `combine_latest`
  there returns a Node.)
